In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import PyPDF2
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from openpyxl import load_workbook
from pyhtml2pdf import converter
import os
import io


---

In [6]:
def create_directory():
    import os
    formatted_list = pd.read_excel(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\eBay formatted list.xlsx', sheet_name='Sheet1', header=None).values.tolist()
    parent_directory = r"\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold"
    for item in formatted_list:
        subfolder_name = item[2]+ ' '+ item[3] + ' (' + str(item[0]) + ' - '+ str(item[1]) + ')'
        subfolder_path = os.path.join(parent_directory, subfolder_name)
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

    vehicles_df = pd.DataFrame(columns=['Search Date', 'Title', 'Sold?', 'Sold Date', 'Link', 'VIN', 'Mileage', 'Condition', 'Year', 'ForSaleBy', 'Trim', 'Transmission', 'Make', 'Model', 'Submodel', 'Price', 'Best Offer Price Override', 'Owners', 'Accidents', 'VehicleTitle'])
    for item in formatted_list:
        subfolder_name = item[2]+ ' '+ item[3] + ' (' + str(item[0]) + ' - '+ str(item[1]) + ')'
        subfolder_path = os.path.join(parent_directory, subfolder_name)
        vehicles_df.to_csv(subfolder_path + r'\listings.csv', index=False)

    parent_directory = r"\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale"
    for item in formatted_list:
        subfolder_name = item[2]+ ' '+ item[3] + ' (' + str(item[0]) + ' - '+ str(item[1]) + ')'
        subfolder_path = os.path.join(parent_directory, subfolder_name)
        if not os.path.exists(subfolder_path):
            os.makedirs(subfolder_path)

    vehicles_df = pd.DataFrame(columns=['Search Date', 'Title', 'Link', 'VIN', 'Mileage', 'Condition', 'Year', 'ForSaleBy', 'Trim', 'Transmission', 'Make', 'Model', 'Submodel', 'Number of Bids', 'Current Bid', 'Buy it Now Price','Owners', 'Accidents', 'VehicleTitle'])
    for item in formatted_list:
        subfolder_name = item[2]+ ' '+ item[3] + ' (' + str(item[0]) + ' - '+ str(item[1]) + ')'
        subfolder_path = os.path.join(parent_directory, subfolder_name)
        vehicles_df.to_csv(subfolder_path + r'\listings.csv', index=False)

In [52]:

def run_sold_search():
    Overall_Data = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\Overall Listings.csv')
    formatted_list = pd.read_excel(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\eBay formatted list.xlsx', sheet_name='Sheet1')
    formatted_list = formatted_list.values.tolist()
    for x in formatted_list:
        Make = x[2]
        Model = x[4]
        Model_name = x[3]
        Low_Year = x[0]
        High_Year = x[1]
        print('Sold - ' + Make, Model_name)
        Year_Range = range(Low_Year, High_Year+1)
        Year_Range = list(Year_Range)
        Year_Range = [str(i) +'%7C'for i in Year_Range]
        Year_Range = ''.join(Year_Range)
        Year_Range = Year_Range[:-3]
        sold_url = "https://www.ebay.com/sch/Cars-Trucks/6001/i.html?makeval="+Make+"&modelval="+Model+"&LH_ItemCondition=3000%7C1000%7C2500&_nkw="+Make+Model+"&LH_Sold=1&LH_Complete=1&rt=nc&LH_PrefLoc=98&UF_single_selection=Make%3A"+Make+"%2CModel%3A"+Model+"&UF_context=finderType%3AVEHICLE_FINDER&_sacat=6001&_stpos=40508&_fspt=1&Model%2520Year="+Year_Range
        vehicles = get_vehicle_sold_info(sold_url)
        model_specific_df_new = pd.DataFrame(vehicles)
        model_specific_df_old = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv')
        model_specific_df = pd.concat([model_specific_df_old, model_specific_df_new], ignore_index=True)
        model_specific_df = model_specific_df.dropna(subset=['VIN'])
        model_specific_df = model_specific_df.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','Price','VehicleTitle','Make','Model'], keep='first')
        model_specific_df.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv', index=False, header=True)
        Overall_Data = pd.concat([Overall_Data, model_specific_df], ignore_index=True)
        Overall_Data = Overall_Data.dropna(subset=['VIN'])
        Overall_Data = Overall_Data.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','Price','VehicleTitle','Make','Model'], keep='first')
        Overall_Data.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\Overall Listings.csv', index=False, header=True)
    Overall_Data = Overall_Data.dropna(subset=['VIN'])
    Overall_Data = Overall_Data.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','Price','VehicleTitle','Make','Model'], keep='first')
    Overall_Data.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\Overall Listings.csv', index=False, header=True)

def run_auction_search():
    Overall_Auction_Data = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\Overall Listings.csv')
    formatted_list = pd.read_excel(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\eBay formatted list.xlsx', sheet_name='Sheet1')
    formatted_list = formatted_list.values.tolist()
    for x in formatted_list:
        Make = x[2]
        Model = x[4]
        Model_name = x[3]
        Low_Year = x[0]
        High_Year = x[1]
        print('Auction - ' + Make, Model_name)
        Year_Range = range(Low_Year, High_Year+1)
        Year_Range = list(Year_Range)
        Year_Range = [str(i) +'%7C'for i in Year_Range]
        Year_Range = ''.join(Year_Range)
        Year_Range = Year_Range[:-3]
        auction_url = "https://www.ebay.com/sch/Cars-Trucks/6001/i.html?makeval="+Make+"&modelval="+Model+"&LH_ItemCondition=3000%7C1000%7C2500&_nkw="+Make+Model+"&UF_single_selection=Make%3A"+Make+"%2CModel%3A"+Model+"&UF_context=finderType%3AVEHICLE_FINDER&_sacat=6001&_stpos=40508&_fspt=1&Model%2520Year="+Year_Range
        vehicles = get_vehicle_auction_info(auction_url)
        model_specific_df_new = pd.DataFrame(vehicles)
        model_specific_df_old = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv')
        model_specific_df = pd.concat([model_specific_df_old, model_specific_df_new], ignore_index=True)
        model_specific_df = model_specific_df.dropna(subset=['VIN'])
        model_specific_df = model_specific_df.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','VehicleTitle','Make','Model'], keep='first')
        model_specific_df.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv', index=False, header=True)
        Overall_Auction_Data = pd.concat([Overall_Auction_Data, model_specific_df], ignore_index=True)
        Overall_Auction_Data = Overall_Auction_Data.dropna(subset=['VIN'])
        Overall_Auction_Data = Overall_Auction_Data.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','VehicleTitle','Make','Model'], keep='first')
        Overall_Auction_Data.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\Overall Listings.csv', index=False, header=True)
    Overall_Auction_Data = Overall_Auction_Data.dropna(subset=['VIN'])
    Overall_Auction_Data = Overall_Auction_Data.drop_duplicates(subset=['Title', 'Mileage','Condition','Year','VIN','VehicleTitle','Make','Model'], keep='first')
    Overall_Auction_Data.to_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\Overall Listings.csv', index=False, header=True)

def get_data(url):
    r = requests.get(url)
    soup = BeautifulSoup(r.text,'html.parser')
    try:
        target= soup.find('li', {'class': 'srp-river-answer srp-river-answer--REWRITE_START'})
        for e in target.find_all_next():
            e.clear()
    except:
        pass
    return soup

def parse(soup):
    listings = []
    results = soup.find_all('div', {'class': 's-item__info clearfix'})
    for item in results:
        product = {
            'title': item.find('div',{'class': 's-item__title'}).text,
            'link': item.find('a',{'class': 's-item__link'})['href']
        }
        listings.append(product)
    
    return listings

def get_vehicle_auction_info(url):
    soup = get_data(url)
    listings = parse(soup)
    vehicles = []
    for listing in listings:
        if 'eBay' in listing['title']:
            del listing
        else:
            now = datetime.now()
            date = now.strftime("%Y-%m-%d")
            url2 = listing['link']
            r2 = requests.get(url2)
            soup2 = BeautifulSoup(r2.text,'html.parser')
            spans = soup2.find_all(lambda tag: tag.name == 'span' and 'ux-textspans' in tag.get('class', []))
            vehicle_data = {
                'Search Date': date,
                'Title': listing['title'],
                'Link': url2,
                'VIN': None,
                'Mileage': None,
                'Condition': None,
                'Year': None,
                'ForSaleBy': None,
                'Trim': None,
                'Transmission': None,
                'Make':None,
                'Model': None,
                'Submodel': None,
                'Number of Bids': None,
                'Current Bid': None,
                'Buy it Now Price': None,
                'Classified Ad Price': None,
                'Owners': None,
                'Accidents': None,
                'VehicleTitle': None,
            }
            for i in range(len(spans)):
                if spans[i].text == "VIN (Vehicle Identification Number)":
                    vehicle_data['VIN'] = spans[i + 1].text
                if spans[i].text == "Mileage":
                    vehicle_data['Mileage'] = spans[i + 1].text
                if spans[i].text == "Condition":
                    vehicle_data['Condition'] = spans[i + 1].text.split(":")[0]
                if spans[i].text == "Year":
                    vehicle_data['Year'] = spans[i + 1].text
                if spans[i].text == "For Sale By":
                    vehicle_data['ForSaleBy'] = spans[i + 1].text
                if spans[i].text == "Trim":
                    vehicle_data['Trim'] = spans[i + 1].text
                if spans[i].text == "Transmission":
                    vehicle_data['Transmission'] = spans[i + 1].text
                if spans[i].text == "Model":
                    vehicle_data['Model'] = spans[i + 1].text
                if spans[i].text == "Sub Model":
                    vehicle_data['Submodel'] = spans[i + 1].text
                if spans[i].text == "Owners":
                    vehicle_data['Owners'] = spans[i + 1].text
                if spans[i].text == "Accidents":
                    vehicle_data['Accidents'] = spans[i + 1].text
                if spans[i].text == "Vehicle Title":
                    vehicle_data['VehicleTitle'] = spans[i + 1].text
                if spans[i].text == "Make":
                    vehicle_data['Make'] = spans[i + 1].text
                if spans[i].text == "Price:":
                    vehicle_data['Buy it Now Price'] = spans[i + 1].text.split("$")[1]
                if spans[i].text == "Current bid:":
                    vehicle_data['Current Bid'] = spans[i + 1].text.split("$")[1]
                    vehicle_data['Number of Bids'] = spans[i + 2].text.split(" ")[0]
                if spans[i].text == "Classified ad price:":
                    vehicle_data['Classified Ad Price'] = spans[i + 1].text.split("$")[1]
            vehicles.append(vehicle_data)
    return vehicles

def get_vehicle_sold_info(url):
    soup = get_data(url)
    listings = parse(soup)
    vehicles = []
    for listing in listings:
        if 'eBay' in listing['title']:
            del listing
        else:
            now = datetime.now()
            date = now.strftime("%Y-%m-%d")
            url2 = listing['link']
            r2 = requests.get(url2)
            soup2 = BeautifulSoup(r2.text,'html.parser')
            spans = soup2.find_all(lambda tag: tag.name == 'span' and 'ux-textspans' in tag.get('class', []))
            vehicle_data = {
                'Search Date': date,
                'Title': listing['title'],
                'Sold?': None,
                'Sold Date': None,
                'Link': url2,
                'VIN': None,
                'Mileage': None,
                'Condition': None,
                'Year': None,
                'ForSaleBy': None,
                'Trim': None,
                'Transmission': None,
                'Make':None,
                'Model': None,
                'Submodel': None,
                'Price': None,
                'Best Offer Price Override': None,
                'Owners': None,
                'Accidents': None,
                'VehicleTitle': None,
            }
            for i in range(len(spans)):
                if spans[i].text == "VIN (Vehicle Identification Number)":
                    vehicle_data['VIN'] = spans[i + 1].text
                if spans[i].text == "Mileage":
                    vehicle_data['Mileage'] = spans[i + 1].text
                if spans[i].text == "Condition":
                    vehicle_data['Condition'] = spans[i + 1].text.split(":")[0]
                if spans[i].text == "Year":
                    vehicle_data['Year'] = spans[i + 1].text
                if spans[i].text == "For Sale By":
                    vehicle_data['ForSaleBy'] = spans[i + 1].text
                if spans[i].text == "Trim":
                    vehicle_data['Trim'] = spans[i + 1].text
                if spans[i].text == "Transmission":
                    vehicle_data['Transmission'] = spans[i + 1].text
                if spans[i].text == "Model":
                    vehicle_data['Model'] = spans[i + 1].text
                if spans[i].text == "Sub Model":
                    vehicle_data['Submodel'] = spans[i + 1].text
                if spans[i].text == "Owners":
                    vehicle_data['Owners'] = spans[i + 1].text
                if spans[i].text == "Accidents":
                    vehicle_data['Accidents'] = spans[i + 1].text
                if spans[i].text == "Vehicle Title":
                    vehicle_data['VehicleTitle'] = spans[i + 1].text
                if spans[i].text == "Make":
                    vehicle_data['Make'] = spans[i + 1].text
                if 'This listing ended' in spans[i].text:
                    vehicle_data['Sold Date'] = spans[i].text.split("on ")[1]
                if spans[i].text == 'SOLD':
                    vehicle_data['Sold?'] = spans[i].text
                if 'Best offer accepted' in spans[i].text:
                    vehicle_data['Best Offer Price Override'] = 'X' 
                if spans[i].text == "Sold for:":
                    vehicle_data['Price'] = spans[i + 1].text.split("$")[1]  
                if "Winning bid" in spans[i].text:
                    vehicle_data['Price'] = spans[i + 1].text.split("$")[1]
            vehicles.append(vehicle_data)
    return vehicles

def add_timestamp_to_pdf(input_pdf_path, output_pdf_path):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(input_pdf_path, 'rb') as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        pdf_writer = PyPDF2.PdfWriter()
        packet = io.BytesIO()
        can = canvas.Canvas(packet, pagesize=letter)
        can.drawString(100, 100, timestamp)  
        can.save()
        packet.seek(0)
        new_pdf = PyPDF2.PdfReader(packet)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            page.merge_page(new_pdf.pages[0])
            pdf_writer.add_page(page)
        with open(output_pdf_path, 'wb') as output_pdf_file:
            pdf_writer.write(output_pdf_file)

def pdfs_and_timestamps():
    formatted_list = pd.read_excel(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\eBay formatted list.xlsx', sheet_name='Sheet1')
    formatted_list = formatted_list.values.tolist()
    for x in formatted_list:
        Make = x[2]
        Model_name = x[3]
        print(Make + ' ' + Model_name+' - PDFs')
        Low_Year = x[0]
        High_Year = x[1]
        vehicle_info = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\Sold\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv')
        vehicle_info = vehicle_info.values.tolist()
        for i in vehicle_info:
            parent_directory = r"\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\PDFs\Sold\\"+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')'
            subfolder_name = i[5]
            try:
                subfolder_path = os.path.join(parent_directory, subfolder_name)
                if not os.path.exists(subfolder_path):
                    os.makedirs(subfolder_path)
                    pdf_path = os.path.join(subfolder_path, subfolder_name)
                    print(Make + ' ' + Model_name+' running converter')
                    converter.convert(i[4], pdf_path+'.pdf')
                    output_pdf_path = pdf_path +'_timestamped.pdf'
                    input_pdf_path = pdf_path +'.pdf'
                    add_timestamp_to_pdf(input_pdf_path, output_pdf_path)
            except:
                pass
        vehicle_info = pd.read_csv(r'\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\Data\For Sale\\'+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')\listings.csv')
        vehicle_info = vehicle_info.values.tolist()
        for i in vehicle_info:
            parent_directory = r"\\na2-fs1\DCDATA6\Cases\Active\BMW Davis (50012)\Research\eBay\PDFs\For Sale\\"+Make+" "+Model_name+" ("+str(Low_Year)+" - "+str(High_Year)+')'
            subfolder_name = i[3]
            try:
                subfolder_path = os.path.join(parent_directory, subfolder_name)
                if not os.path.exists(subfolder_path):
                    os.makedirs(subfolder_path)
                    pdf_path = os.path.join(subfolder_path, subfolder_name)
                    print(Make + ' ' + Model_name+' running converter')
                    converter.convert(i[2], pdf_path+'.pdf')
                    output_pdf_path = pdf_path +'_timestamped.pdf'
                    input_pdf_path = pdf_path +'.pdf'
                    add_timestamp_to_pdf(input_pdf_path, output_pdf_path)
            except:
                pass

def run_search():
    run_sold_search()
    run_auction_search()
    pdfs_and_timestamps()


In [53]:
pdfs_and_timestamps()

BMW 550i - PDFs
BMW 550i running converter
BMW 550i running converter
BMW 550i running converter
BMW 550i xDrive - PDFs
BMW 550i GT - PDFs
BMW 550i GT xDrive - PDFs
BMW 650i Gran Coupe - PDFs
BMW 650i xDrive Gran Coupe - PDFs
BMW 650Ci - PDFs
BMW 650Ci xDrive  - PDFs
BMW 750i - PDFs
BMW 750i running converter
BMW 750i running converter
BMW 750i running converter
BMW 750i running converter
BMW 750Li - PDFs
BMW 750Li running converter
BMW 750Li running converter
BMW 750Li running converter
BMW 750Li running converter
BMW 750Li running converter
BMW 750i xDrive - PDFs
BMW 750Li xDrive - PDFs
BMW 760Li - PDFs
BMW Alpina B7 - PDFs
BMW ActiveHybrid 7 - PDFs
BMW X5  - PDFs
BMW X6 - PDFs
BMW X6 running converter
BMW X6 running converter
BMW X6 running converter
Mini Cooper - PDFs
Mini Cooper running converter
Mini Cooper running converter
Mini Cooper running converter
Mini Cooper running converter
Mini Cooper running converter
Mini Cooper running converter
Mini Cooper running converter
Mini Co